# Coleta dados apifootball
## Etapa coleta dos dados e gravação na bronze
Esta etapa consiste em fazer uma consulta e gravar os dados em uma camada bronze, adicionando apenas um campo para identificar a data e hora da coleta.

A coleta ocorre por meio da API https://apifootball.com/.

O método usado é o get_events, documentado [aqui](https://apifootball.com/documentation/#Events).

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import lit
from datetime import datetime, timedelta

In [0]:
api_key = dbutils.fs.head("dbfs:/Volumes/apifootball/bronze/arquivos/apifootball_key.txt").strip()
#api_key = (spark.read.text("/Volumes/apifootball/bronze/arquivos/apifootball_key.txt").first()[0].strip())
print(api_key) #remover posteriormente

In [0]:
%skip
url = (
    f'https://apiv3.apifootball.com/'
    f'?action=get_events'
    f'&from=2026-01-01'
    f'&to=2026-12-31'
    f'&league_id=99' #brasil
    f'&APIkey={api_key}'
)

# Request
response = requests.get(url)

# Status
print(response.status_code)

# JSON
data = response.json()

# DataFrame
df_raw = pd.DataFrame(data)

dh_processing_br = datetime.now() - timedelta(hours=3)
df_raw = df_raw.withColumn('dh_processing_br', lit(dh_processing_br))
#df_raw.createOrReplaceTempView('vw_raw')

# Visualizar
print(df_raw.shape)

In [0]:
#aqui depois será a própria api
dh_processing_br = datetime.now() - timedelta(hours=3)
df_raw = spark.read.table("apifootball.bronze.serie_a_raw").withColumn("dh_processing_br", lit(dh_processing_br))
#display(df_raw)

In [0]:
# pelo o que vi no google collab, a API puxa tudo como string, acho que podemos deixar assim e até forçar que seja assim
# na silver tratamos isso
df_raw.printSchema()

In [0]:
%sql
-- aqui optei por fazer em sql por ser visualmente mais fácil de ler, mas deixei o código em pyspark para explorar
-- ou posso tirar e consumir o max na silver
delete from apifootball.bronze.matches_raw as mr
where cast(mr.dh_processing_br as date) in (select cast(d.dh_processing_br as date) from vw_raw as d)

In [0]:
%skip
from delta.tables import DeltaTable

df_raw.createOrReplaceTempView("vw_raw")
# Datas presentes na nova carga
datas = [
    row.dt
    for row in vw_raw.selectExpr(
        "cast(dh_processing_br as date) as dt"
    ).distinct().collect()
]

delta_table = DeltaTable.forName(
    spark,
    "apifootball.bronze.matches_raw"
)

delta_table.delete(
    condition=F.col("dh_processing_br").cast("date").isin(datas)
)

# Insere a nova carga
vw_raw.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("apifootball.bronze.matches_raw")

In [0]:
%sql
delete from apifootball.bronze.matches_raw as mr
where year(mr.dh_processing_br) < (year(current_date)-2)

In [0]:
df_raw.write.mode("append").saveAsTable("apifootball.bronze.matches_raw")

In [0]:
spark.read.table("apifootball.bronze.matches_raw").display()